# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

## 1. Ranked actions + reason codes

The model output (Week 5, grouped split, AUC-PR ~0.52) identifies pages with elevated probability of acquiring AI-assistant referral traffic. This section turns that score into concrete actions for content teams.

### Action tiers:

**TIER 1: REVIEW & OPTIMIZE** (top 10% of ranked queue)
- **Who:** Editorial + SEO team
- **Action:** Deep-dive audit of these pages for AI-citation readiness
- **Reason codes:**
  - `high_engagement_recent_update`: High user engagement + actively maintained → best candidates for AI optimization
  - `strong_signal_ensemble`: Multiple positive signals (engagement + position + volume)
- **Expected outcome:** Pages in this tier are most likely to attract AI traffic; editorial polish can accelerate it
- **Human involvement:** Required (see Section 3)

**TIER 2: MONITOR & PREPARE** (10–30% of ranked queue)
- **Who:** Content team (async, lower priority)
- **Action:** Flag for next quarter's content calendar; prepare for refresh
- **Reason codes:**
  - `moderate_engagement_stale`: Good engagement but last updated >30 days ago
  - `high_volume_potential`: Page has search volume but not yet breaking through
- **Expected outcome:** These pages may become Tier 1 after a small refresh
- **Human involvement:** Optional (flag + schedule)

**TIER 3: DEFER & RECHECK** (bottom 60%)
- **Who:** Monitoring bot (async)
- **Action:** Re-check quarterly; escalate only if signals improve
- **Reason codes:**
  - `low_engagement_signal`: Engagement < 1%; not yet visitor-ready
  - `new_content`: Recent pages; waiting for traffic signal
  - `low_priority`: Not enough data to recommend
- **Expected outcome:** Most will remain low-priority; ~5–10% escalate per quarter
- **Human involvement:** None unless flagged by retrain trigger

### Decision rule (from Week 5 model):
score = engagement_rate * (1 if days_since_last_update <= 30 else 0.5) * position_weight action = "REVIEW" if score >= 75th percentile = "MONITOR" if score >= 50th percentile = "DEFER" else reason_code = [determined by feature combination above]

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import json

print("="*70)
print("SECTION 1: RANKED ACTIONS + REASON CODES")
print("="*70)

# Clone repo if needed (Colab)
if not os.path.exists('flyrank-internship-w1'):
    !git clone https://github.com/hanaahmedradwan123-commits/flyrank-internship-w1.git
    os.chdir('flyrank-internship-w1')

# Load data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create target
df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)

# Remove leaky columns
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

# Features
features = ['engagement_rate', 'days_since_last_update', 'word_count', 'ctr', 'avg_position']
X = df[features].fillna(0)
y = df['has_ai_traffic']

# Grouped split by client (from Week 6 audit)
unique_clients = df['client_id'].unique()
n_clients = len(unique_clients)
n_train_clients = int(0.7 * n_clients)

train_clients = np.random.RandomState(42).choice(unique_clients, size=n_train_clients, replace=False)
test_clients = np.setdiff1d(unique_clients, train_clients)

train_mask = df['client_id'].isin(train_clients)
test_mask = df['client_id'].isin(test_clients)

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

# Train model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Generate scores for the entire dataset (for ranking)
X_all_scaled = scaler.transform(X)
scores = model.predict_proba(X_all_scaled)[:, 1]

df['model_score'] = scores

# Create reason codes
df['reason_code'] = 'low_engagement_signal'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] <= 30), 'reason_code'] = 'high_engagement_recent_update'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] > 30), 'reason_code'] = 'moderate_engagement_stale'
df.loc[(df['word_count'] > 2000) & (df['ctr'] > 0.5), 'reason_code'] = 'strong_signal_ensemble'
df.loc[(df['engagement_rate'] < 1.0), 'reason_code'] = 'low_engagement_signal'

# Create action tiers based on percentiles
p75 = df['model_score'].quantile(0.75)
p50 = df['model_score'].quantile(0.50)

df['action'] = 'DEFER'
df.loc[df['model_score'] >= p75, 'action'] = 'REVIEW'
df.loc[(df['model_score'] >= p50) & (df['model_score'] < p75), 'action'] = 'MONITOR'

# Sort by score
df_ranked = df.sort_values('model_score', ascending=False)

print(f"\n✓ Model trained on grouped split")
print(f"  Train clients: {len(train_clients)}")
print(f"  Test clients: {len(test_clients)}")
print(f"\nAction distribution:")
print(df_ranked['action'].value_counts())
print(f"\nReason code distribution:")
print(df_ranked['reason_code'].value_counts())

print(f"\nTop 5 actions:")
output_cols = ['content_id', 'model_score', 'action', 'reason_code', 'engagement_rate', 'days_since_last_update']
print(df_ranked[output_cols].head(10).to_string(index=False))

SECTION 1: RANKED ACTIONS + REASON CODES
Cloning into 'flyrank-internship-w1'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 158 (delta 62), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.88 MiB | 10.36 MiB/s, done.
Resolving deltas: 100% (62/62), done.

✓ Model trained on grouped split
  Train clients: 22
  Test clients: 10

Action distribution:
action
DEFER      15000
REVIEW      7500
MONITOR     7500
Name: count, dtype: int64

Reason code distribution:
reason_code
low_engagement_signal            25151
high_engagement_recent_update     2202
strong_signal_ensemble            1465
moderate_engagement_stale         1182
Name: count, dtype: int64

Top 5 actions:
          content_id  model_score action                   reason_code  engagement_rate  days_since_last_update
content_cfceaeb2ffa1     0.204344 REVIEW high_engagement_recent_update    

## 2. Intended use and limits

### Intended use:

**Primary:** Editorial team decides which pages to audit for AI-citation readiness — not to automate decisions, but to prioritize human review.

**Secondary:** Content calendar planning — flag pages that may be ready for refresh or optimization work.

**Not intended for:**
- Automated content deletion or demotion
- Real-time ranking changes without human approval
- Deployment to production search/recommendation systems (this is a guidance playbook, not a deployed model)

### Who?
- **Tier 1 (REVIEW):** Editorial + SEO lead
- **Tier 2 (MONITOR):** Content planner (async)
- **Tier 3 (DEFER):** Monitoring system only

### What the model does:
- Assigns each page a probability (0–1) of acquiring AI-assistant referral traffic
- Ranks pages by this probability
- Assigns reason codes to explain the score (engagement, freshness, etc.)

### What the model does NOT do:
- Predict future growth or revenue impact
- Guarantee that optimizing a page will result in AI traffic (correlation ≠ causation)
- Account for brand fit, audience match, or competitive context
- Replace human editorial judgment

### Valid time window:
This playbook is built on 90-day aggregate metrics. Use it for decisions within ~30 days of model training. Beyond 30 days, retrain (see Section 4).

### Performance reality (from Week 6 audit):
- **On grouped split (new clients):** Precision@50 ≈ 0.42 (42% of top-50 pages actually acquired AI traffic in test window)
- **Random split (had leakage):** Precision@50 ≈ 0.68 (overstated)
- **Honest expectation:** 4–5 of the top 10 pages will likely have AI-traffic opportunity

In [2]:
print("\n" + "="*70)
print("SECTION 2: INTENDED USE & LIMITS")
print("="*70)

print("""
INTENDED USE CHECKLIST:
✓ Editorial team prioritizes which pages to audit
✓ Content planner flags pages for next calendar
✓ Monitoring system re-checks Tier 3 quarterly
✗ Do NOT automate content decisions without human approval
✗ Do NOT deploy to production ranking without A/B test
✗ Do NOT guarantee revenue impact

VALID USE CASES:
• "Which 20 pages should we review for AI optimization this sprint?"
  → Answer: Top 20 from REVIEW tier

• "Which pages are ready for a content refresh?"
  → Answer: MONITOR tier with engagement > 3% and last update > 60d

• "How often should we retrain?"
  → Answer: Monthly for engagement_rate shifts; quarterly for trend validation

INVALID USE CASES:
• "Automate which pages to show first in search"
  → Why: Grouped split shows model learns client patterns; untested on real ranking impact

• "Predict how much traffic this page will gain"
  → Why: Model predicts probability of AI traffic, not quantity

• "Decide which pages to delete"
  → Why: No evidence of downside; ethical gate needed before any action
""")

# Compute and display limits
precision_at_50_honest = 0.42  # From Week 6 audit
recall_at_50 = 0.10  # Assuming 500 total positive examples

print(f"\nPERFORMANCE LIMITS (from Week 6 grouped split):")
print(f"  Precision@50: {precision_at_50_honest:.2%} (4–5 of top 10 likely AI-ready)")
print(f"  Model learned patterns from {len(train_clients)} seen clients")
print(f"  May underperform on new clients or time periods")
print(f"\nNOT VALID FOR:")
print(f"  • Real-time decisions without 24-hour latency")
print(f"  • Personalized recommendations (no user context)")
print(f"  • Competitive intelligence or pricing decisions")


SECTION 2: INTENDED USE & LIMITS

INTENDED USE CHECKLIST:
✓ Editorial team prioritizes which pages to audit
✓ Content planner flags pages for next calendar
✓ Monitoring system re-checks Tier 3 quarterly
✗ Do NOT automate content decisions without human approval
✗ Do NOT deploy to production ranking without A/B test
✗ Do NOT guarantee revenue impact

VALID USE CASES:
• "Which 20 pages should we review for AI optimization this sprint?"
  → Answer: Top 20 from REVIEW tier
  
• "Which pages are ready for a content refresh?"
  → Answer: MONITOR tier with engagement > 3% and last update > 60d

• "How often should we retrain?"
  → Answer: Monthly for engagement_rate shifts; quarterly for trend validation

INVALID USE CASES:
• "Automate which pages to show first in search"
  → Why: Grouped split shows model learns client patterns; untested on real ranking impact
  
• "Predict how much traffic this page will gain"
  → Why: Model predicts probability of AI traffic, not quantity
  
• "Decide whi

## 3. Human review + the no-go list

### Before any action, a human must verify:

#### TIER 1 (REVIEW) — Editorial must check:
1. **Content quality:** Does the page meet our quality bar? (Read first 2 paragraphs; spot-check facts)
2. **Audience fit:** Is this content a good fit for AI citations? (Does it answer common questions clearly?)
3. **Competitive context:** Are we swimming upstream or riding a wave? (Quick search for top 3 competitors)
4. **Brand alignment:** Does this page represent our brand voice? (Tone, expertise, trustworthiness)
5. **Sensitivity gate:** Does this touch on health, finance, legal, or politics? → Escalate to Legal/Compliance
6. **Recency check:** Is the publication date accurate? Did we check the data currency? → Verify timestamps

#### TIER 2 (MONITOR) — Content team can auto-flag if:
1. Last update > 90 days old (flag for refresh)
2. Engagement plateaued for 2 consecutive quarters (flag for rewrite)
3. Competitive pages improved significantly (flag for our relaunch)

#### TIER 3 (DEFER) — No human action needed; automated monitoring only

### NO-GO cases (do NOT recommend):

- **Fresh content (<7 days old):** Not enough visitor data yet; wait 2 weeks
- **Unverified facts:** Any page with outdated sources or unconfirmed claims → Hold until fact-check complete
- **Sensitive topics without clear sourcing:** Health, finance, legal — only after Legal review
- **Content in active litigation or complaint:** No action until resolved
- **Pages with 0 impressions:** Invisible to search; no AI traffic likely until visibility improves first
- **Brand-misaligned pages:** Even if high-engagement, do NOT recommend if tone/voice conflicts with brand
- **Highly seasonal content:** (e.g., holiday guides in March) → Rerank quarterly, not by static score

### Human review workflow:

Model scores page → Assigned to Tier 1 (REVIEW) ↓ Editorial checks 6 boxes above + flags sensitivity gate ↓ If all clear → "greenlit for AI optimization focus" ↓ If any gate fails → "hold for [reason]" → re-check next quarter

In [3]:
print("\n" + "="*70)
print("SECTION 3: HUMAN REVIEW + NO-GO LIST")
print("="*70)

# Create a review checklist
review_checklist = {
    'Content quality': ['Read first 2 paragraphs', 'Spot-check facts', 'Verify citations'],
    'Audience fit': ['Does it answer common questions?', 'Clear structure?', 'AI-friendly format?'],
    'Competitive context': ['Top 3 competitors rank where?', 'Our angle is unique?'],
    'Brand alignment': ['Tone matches?', 'Expertise evident?', 'Trustworthy?'],
    'Sensitivity gate': ['Health/finance/legal?', 'Escalate if yes'],
    'Recency check': ['Publication date correct?', 'Data currency OK?']
}

print("\nREVIEW CHECKLIST FOR TIER 1 (REVIEW) PAGES:")
for category, checks in review_checklist.items():
    print(f"\n  {category}:")
    for check in checks:
        print(f"    [ ] {check}")

print("\n" + "="*70)
print("NO-GO CASES (Do NOT recommend):")
print("="*70)

no_go_reasons = [
    'Fresh content (<7 days) — not enough visitor data yet',
    'Unverified facts — hold until fact-check complete',
    'Sensitive topics (health/finance/legal) — require Legal review first',
    'Pages in active litigation — wait until resolved',
    '0 impressions — invisible to search; improve visibility first',
    'Brand-misaligned pages — even if high-engagement',
    'Highly seasonal content — rerank quarterly',
]

for i, reason in enumerate(no_go_reasons, 1):
    print(f"  {i}. {reason}")

print("\n" + "="*70)
print("FLAGGING RULES:")
print("="*70)

# Flag pages that should NOT be recommended
df_ranked['review_flag'] = ''
df_ranked.loc[df_ranked['content_age_days'] < 7, 'review_flag'] = 'HOLD: Fresh content (<7d)'
df_ranked.loc[df_ranked['impressions_90d'] == 0, 'review_flag'] = 'HOLD: Zero impressions'

flagged_count = (df_ranked['review_flag'] != '').sum()
print(f"\nPages flagged for human review gates: {flagged_count}")
print(f"\nExample flagged pages:")
if flagged_count > 0:
    print(df_ranked[df_ranked['review_flag'] != ''][['content_id', 'action', 'review_flag']].head(5).to_string(index=False))


SECTION 3: HUMAN REVIEW + NO-GO LIST

REVIEW CHECKLIST FOR TIER 1 (REVIEW) PAGES:

  Content quality:
    [ ] Read first 2 paragraphs
    [ ] Spot-check facts
    [ ] Verify citations

  Audience fit:
    [ ] Does it answer common questions?
    [ ] Clear structure?
    [ ] AI-friendly format?

  Competitive context:
    [ ] Top 3 competitors rank where?
    [ ] Our angle is unique?

  Brand alignment:
    [ ] Tone matches?
    [ ] Expertise evident?
    [ ] Trustworthy?

  Sensitivity gate:
    [ ] Health/finance/legal?
    [ ] Escalate if yes

  Recency check:
    [ ] Publication date correct?
    [ ] Data currency OK?

NO-GO CASES (Do NOT recommend):
  1. Fresh content (<7 days) — not enough visitor data yet
  2. Unverified facts — hold until fact-check complete
  3. Sensitive topics (health/finance/legal) — require Legal review first
  4. Pages in active litigation — wait until resolved
  5. 0 impressions — invisible to search; improve visibility first
  6. Brand-misaligned pages 

## 4. Monitoring / retrain triggers

### How to know the playbook went stale:

#### Monthly checks (automated):
- [ ] **Engagement shift:** If mean engagement_rate dropped >20% from training → Retrain
- [ ] **Freshness shift:** If median days_since_last_update increased >30 days → Retrain
- [ ] **New content surge:** If >20% of pages are <7 days old → Retrain
- [ ] **Precision decay:** If Tier 1 pages stop acquiring AI traffic (track for 2 weeks) → Retrain

#### Quarterly reviews (manual):
- [ ] **Editorial feedback:** Are Tier 1 pages actually worth the effort? (Qualitative)
- [ ] **Competitive landscape:** Did competitors publish major updates? → Refresh baseline
- [ ] **Traffic patterns:** Did AI-traffic seasonality appear? (e.g., higher in Q4) → Adjust thresholds
- [ ] **Business objectives:** Did company priorities shift? (e.g., new market entry) → Retrain with new signals

#### Retrain if ANY of:
1. Engagement shift >20%
2. Freshness shift >30 days
3. Precision@50 drops below 0.30 (from 0.42 baseline)
4. New signal appears (e.g., brand mentions, backlinks)
5. Editorial consensus that recommendations are no longer useful

### Retrain workflow:
Trigger fires (e.g., engagement_rate mean dropped 22%) ↓ Pull new data (last 30 days of metrics) ↓ Re-run Week 5 model with new data ↓ Compare new Precision@50 to 0.42 baseline ↓ If new precision >= 0.35 → Deploy new playbook ↓ If new precision < 0.35 → Investigate (leakage? missing signal?) before deploying

Code

### What NOT to monitor:
- Individual page rank or score (too noisy; monitor aggregates instead)
- Exact Tier membership (boundaries are soft; focus on Precision@50 instead)
- Single-page conversion metrics (causality unclear; need A/B test)

In [5]:
print("\n" + "="*70)
print("SECTION 4: MONITORING & RETRAIN TRIGGERS")
print("="*70)

# Simulate monitoring dashboard
print("\nMONTHLY MONITORING METRICS:")

# Baseline from training data
baseline_engagement = df[train_mask]['engagement_rate'].mean()
baseline_freshness = df[train_mask]['days_since_last_update'].mean()
baseline_precision_at_50 = 0.42  # From Week 6

# Current snapshot (simulated as new data)
current_engagement = df['engagement_rate'].mean()
current_freshness = df['days_since_last_update'].mean()

engagement_shift = (current_engagement - baseline_engagement) / baseline_engagement * 100
freshness_shift = current_freshness - baseline_freshness

print(f"  Baseline engagement: {baseline_engagement:.2%}")
print(f"  Current engagement: {current_engagement:.2%}")
print(f"  Shift: {engagement_shift:+.1f}%")
print(f"  → Trigger if shift > ±20%: {'⚠️  YES' if abs(engagement_shift) > 20 else '✓ OK'}")

print(f"\n  Baseline freshness (days since update): {baseline_freshness:.0f}d")
print(f"  Current freshness: {current_freshness:.0f}d")
print(f"  Shift: {freshness_shift:+.0f}d")
print(f"  → Trigger if shift > ±30d: {'⚠️  YES' if abs(freshness_shift) > 30 else '✓ OK'}")

# Check for retrain triggers
triggers = []
if abs(engagement_shift) > 20:
    triggers.append(f"Engagement shift: {engagement_shift:+.1f}%")
if abs(freshness_shift) > 30:
    triggers.append(f"Freshness shift: {freshness_shift:+.0f}d")

print(f"\n{'='*70}")
print("RETRAIN STATUS:")
print(f"{'='*70}")
if triggers:
    print(f"⚠️  {len(triggers)} trigger(s) active:")
    for trigger in triggers:
        print(f"    • {trigger}")
    print(f"\n→ RECOMMENDATION: Retrain model with new data")
else:
    print(f"✓ All metrics stable. No immediate retrain needed.")
    print(f"  Next review: 30 days")

# Export monitoring checklist
monitoring_checklist = {
    'Metric': ['Engagement shift', 'Freshness shift', 'New content surge', 'Precision decay', 'Editorial feedback', 'Competitive check'],
    'Frequency': ['Monthly', 'Monthly', 'Monthly', 'Monthly', 'Quarterly', 'Quarterly'],
    'Threshold': ['±20%', '±30d', '>20% <7d old', '<0.30 Precision@50', 'Consensus', 'Major update'],
    'Action if triggered': ['Retrain', 'Retrain', 'Retrain', 'Retrain', 'Adjust thresholds', 'Refresh baseline']
}

monitoring_df = pd.DataFrame(monitoring_checklist)
print(f"\n{'='*70}")
print("FULL MONITORING CHECKLIST:")
print(f"{'='*70}")
print(monitoring_df.to_string(index=False))


SECTION 4: MONITORING & RETRAIN TRIGGERS

MONTHLY MONITORING METRICS:
  Baseline engagement: 256.01%
  Current engagement: 253.45%
  Shift: -1.0%
  → Trigger if shift > ±20%: ✓ OK

  Baseline freshness (days since update): 44d
  Current freshness: 46d
  Shift: +2d
  → Trigger if shift > ±30d: ✓ OK

RETRAIN STATUS:
✓ All metrics stable. No immediate retrain needed.
  Next review: 30 days

FULL MONITORING CHECKLIST:
            Metric Frequency          Threshold Action if triggered
  Engagement shift   Monthly               ±20%             Retrain
   Freshness shift   Monthly               ±30d             Retrain
 New content surge   Monthly       >20% <7d old             Retrain
   Precision decay   Monthly <0.30 Precision@50             Retrain
Editorial feedback Quarterly          Consensus   Adjust thresholds
 Competitive check Quarterly       Major update    Refresh baseline


## 5. Exports for the paper

This section generates three files for next week's capstone paper:

1. **playbook_queue.csv** — Full ranked queue with scores, actions, reason codes
2. **playbook_summary.json** — Metadata: precision, recall, decision thresholds, no-go rules
3. **action_distribution.csv** — Summary of Tier 1/2/3 distribution and characteristics

These are the artifacts your paper will cite: "See playbook export in work/outputs/"

In [7]:
print("\n" + "="*70)
print("SECTION 5: EXPORTS FOR THE PAPER")
print("="*70)

# Create output directory
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Export full ranked queue
output_queue = df_ranked[['content_id', 'client_id', 'model_score', 'action', 'reason_code',
                           'engagement_rate', 'days_since_last_update', 'word_count', 'ctr', 'avg_position']].copy()
output_queue.to_csv('work/outputs/playbook_queue.csv', index=False)
print("✓ Exported: work/outputs/playbook_queue.csv")
print(f"  Rows: {len(output_queue)}")
print(f"  Columns: {list(output_queue.columns)}")

# 2. Export summary metadata (convert numpy types to Python native types)
summary = {
    'model_type': 'Logistic Regression',
    'split_type': 'Grouped by client (honest validation)',
    'precision_at_50': float(baseline_precision_at_50),
    'total_pages': int(len(df_ranked)),
    'tier_distribution': {
        'REVIEW': int((df_ranked['action'] == 'REVIEW').sum()),
        'MONITOR': int((df_ranked['action'] == 'MONITOR').sum()),
        'DEFER': int((df_ranked['action'] == 'DEFER').sum()),
    },
    'features_used': features,
    'decision_thresholds': {
        'review_percentile': 0.75,
        'monitor_percentile': 0.50,
    },
    'no_go_rules': [
        'Fresh content (<7 days)',
        'Zero impressions',
        'Unverified facts',
        'Sensitive topics without Legal review',
        'Brand-misaligned pages',
    ],
    'retrain_triggers': [
        'Engagement shift >±20%',
        'Freshness shift >±30 days',
        'Precision@50 < 0.30',
        'New major signal available',
    ],
    'valid_use_window': '30 days from training date',
    'training_data': {
        'train_clients': int(len(train_clients)),
        'test_clients': int(len(test_clients)),
        'total_clients': int(n_clients),
    },
    'limitations': [
        'Grouped split shows model learns client patterns; may underperform on entirely new clients',
        'No causal inference; correlation only',
        'Does not predict quantity of traffic, only probability of any traffic',
        'Built on 90-day aggregate metrics; not suitable for real-time decisions',
    ]
}

with open('work/outputs/playbook_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✓ Exported: work/outputs/playbook_summary.json")

# 3. Export action distribution
action_dist = df_ranked.groupby('action').agg({
    'engagement_rate': ['mean', 'std'],
    'days_since_last_update': ['mean', 'std'],
    'model_score': ['mean', 'min', 'max'],
    'content_id': 'count'
}).round(3)
action_dist.columns = ['_'.join(col).strip() for col in action_dist.columns.values]
action_dist.to_csv('work/outputs/action_distribution.csv')
print(f"\n✓ Exported: work/outputs/action_distribution.csv")

# Print summary
print(f"\n{'='*70}")
print("EXPORT SUMMARY FOR PAPER:")
print(f"{'='*70}")
print(f"\nAll files live in work/outputs/")
print(f"  • playbook_queue.csv — {len(output_queue)} pages ranked")
print(f"  • playbook_summary.json — metadata + decision rules")
print(f"  • action_distribution.csv — Tier characteristics")
print(f"\nExample rows from playbook_queue.csv:")
print(output_queue.head(10).to_string(index=False))

print(f"\n{'='*70}")
print("PAPER CITATION TEMPLATE:")
print(f"{'='*70}")
print(f"""
"The validated model (grouped split, Precision@50=0.42) identified {summary['total_pages']} pages
with elevated AI-traffic probability. A human-reviewed playbook (work/outputs/playbook_queue.csv)
assigns each page to one of three action tiers: REVIEW ({summary['tier_distribution']['REVIEW']} pages),
MONITOR ({summary['tier_distribution']['MONITOR']} pages), or DEFER ({summary['tier_distribution']['DEFER']} pages).
See playbook_summary.json for decision rules and no-go gates."
""")


SECTION 5: EXPORTS FOR THE PAPER
✓ Exported: work/outputs/playbook_queue.csv
  Rows: 30000
  Columns: ['content_id', 'client_id', 'model_score', 'action', 'reason_code', 'engagement_rate', 'days_since_last_update', 'word_count', 'ctr', 'avg_position']

✓ Exported: work/outputs/playbook_summary.json

✓ Exported: work/outputs/action_distribution.csv

EXPORT SUMMARY FOR PAPER:

All files live in work/outputs/
  • playbook_queue.csv — 30000 pages ranked
  • playbook_summary.json — metadata + decision rules
  • action_distribution.csv — Tier characteristics

Example rows from playbook_queue.csv:
          content_id         client_id  model_score action                   reason_code  engagement_rate  days_since_last_update  word_count   ctr  avg_position
content_cfceaeb2ffa1 client_7f2253d7e2     0.204344 REVIEW high_engagement_recent_update            100.0                      20      4169.0  0.00           0.1
content_76b07f20b83c client_9f14025af0     0.170471 REVIEW        strong_sign

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.